# Prática — SVM

**Problema:** prever a sobrevivência de passageiros do Titanic e comparar SVM com Random Forest.

Todas as lacunas estão preenchidas. Execute as células de cima para baixo com `Shift + Enter`.

A preparação original não foi fornecida. Nesta versão, uso seis características, validação cruzada estratificada com cinco divisões e semente 42. Os resultados podem ser diferentes dos obtidos na preparação da aula.

## 0. Preparação

Base: [Titanic, disponibilizada pelo projeto Seaborn](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv). O arquivo `titanic.csv` acompanha o notebook.

As entradas são classe, sexo, idade, irmãos/cônjuges, pais/filhos e tarifa. O alvo é `survived`: 0 significa que morreu e 1 significa que sobreviveu. A coluna `alive` não entra no modelo, pois revelaria a resposta.

O sexo é convertido para números. As idades ausentes são preenchidas pela mediana **somente dos dados de treino de cada divisão**, por meio de um pipeline. O escalonamento também é ajustado apenas no treino. Assim, a validação não participa da preparação do modelo.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

passageiros = pd.read_csv(Path("titanic.csv"))
X = passageiros[["pclass", "sex", "age", "sibsp", "parch", "fare"]].copy()
X["sex"] = X["sex"].map({"male": 0, "female": 1})
y = passageiros["survived"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = []

def avaliar(nome, modelo):
    pipeline = make_pipeline(SimpleImputer(strategy="median"), modelo)
    pontuacoes = cross_validate(
        pipeline, X, y, cv=cv, scoring="accuracy", return_train_score=True
    )
    resultado = {
        "Modelo": nome,
        "Treino": pontuacoes["train_score"].mean(),
        "Validação": pontuacoes["test_score"].mean()
    }
    resultados.append(resultado)
    print(f"{nome} | Treino: {resultado['Treino']:.4%} | Validação: {resultado['Validação']:.4%}")

assert len(passageiros) == 891
print(f"{len(passageiros)} passageiros | {y.mean():.0%} sobreviveram")

891 passageiros | 38% sobreviveram


## Etapa 0 — Random Forest de referência

### 0.1 Modelo com 300 árvores e profundidade máxima 6

In [2]:
avaliar("Random Forest (referência)", RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42))

Random Forest (referência) | Treino: 87.5140% | Validação: 83.2747%


## Etapa 1 — O SVM e a escala dos dados

### 1.1 Escalas das colunas

In [3]:
X.describe().loc[["min", "max"]]

,pclass,sex,age,sibsp,parch,fare
min,1.0,0.0,0.42,0.0,0.0,0.0000
max,3.0,1.0,80.00,8.0,6.0,512.3292


### 1.2 SVM sem escala

In [4]:
avaliar("SVM sem escala", SVC())

SVM sem escala | Treino: 68.2097% | Validação: 67.7867%


### 1.3 SVM com escala

In [5]:
avaliar("SVM com escala", make_pipeline(StandardScaler(), SVC()))

SVM com escala | Treino: 83.5296% | Validação: 82.4895%


### 1.4 Quanto a validação mudou? Qual coluna atrapalhava?

A validação passou de 67,8% para 82,5%, uma melhora de 14,7 pontos percentuais. A tarifa (`fare`) provavelmente atrapalhava mais, porque chega a cerca de 512, enquanto sexo varia apenas entre 0 e 1. Sem escala, essa diferença pesa muito nas distâncias.

## Etapa 2 — O hiperparâmetro C

### 2.1 C muito baixo e muito alto

In [6]:
avaliar("SVM C=0.01", make_pipeline(StandardScaler(), SVC(C=0.01)))
avaliar("SVM C=100", make_pipeline(StandardScaler(), SVC(C=100)))

SVM C=0.01 | Treino: 61.6162% | Validação: 61.6163%


SVM C=100 | Treino: 86.1952% | Validação: 80.5825%


### 2.2 O que representa uma validação perto de 62%?

É aproximadamente a proporção de passageiros que morreram: 549 de 891, ou 61,6%. Um modelo que sempre previsse que ninguém sobreviveu já acertaria essa porcentagem. Com C=0.01, o resultado ficou praticamente nesse nível.

### 2.3 O que aconteceu com a distância entre treino e validação com C=100?

A diferença aumentou. Com C=100, o treino ficou em 86,2% e a validação em 80,6%, uma distância de 5,6 pontos percentuais. Com C=1, essa distância era de 1,0 ponto. Isso indica maior tendência a overfitting.

## Etapa 3 — Kernel e gamma

### 3.1 Kernel linear

In [7]:
avaliar("SVM kernel linear", make_pipeline(StandardScaler(), SVC(kernel="linear")))

SVM kernel linear | Treino: 78.6756% | Validação: 78.6755%


### 3.2 Gamma alto

In [8]:
avaliar("SVM gamma=10", make_pipeline(StandardScaler(), SVC(gamma=10)))

SVM gamma=10 | Treino: 89.8427% | Validação: 76.2055%

### 3.3 O RBF foi melhor que o linear? O que isso sugere?

Sim. O RBF alcançou 82,5% na validação, contra 78,7% do linear. Isso sugere que uma fronteira curva representa melhor as relações entre as características e a sobrevivência nessa preparação.

### 3.4 O que aconteceu com treino e validação com gamma=10?

O treino subiu para 89,8%, mas a validação caiu para 76,2%. O modelo ficou mais ajustado aos dados de treino e generalizou pior, o que é um sinal de overfitting.

## Desafio — Busca automática de hiperparâmetros

A grade combina três valores de C e três valores de gamma: nove combinações, cada uma avaliada nas mesmas cinco divisões. O preenchimento de dados ausentes faz parte do pipeline para evitar vazamento de informação.

In [9]:
grade = {
    "svc__C": [0.1, 1, 10],
    "svc__gamma": [0.01, 0.1, 1]
}
busca = GridSearchCV(
    make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), SVC()),
    grade,
    cv=cv,
    scoring="accuracy",
    return_train_score=True
)
busca.fit(X, y)

print("Melhor combinação:", busca.best_params_)
print(f"Validação: {busca.best_score_:.4%}")
resultados.append({
    "Modelo": "SVM ajustado (GridSearchCV)",
    "Treino": busca.cv_results_["mean_train_score"][busca.best_index_],
    "Validação": busca.best_score_
})

Melhor combinação: {'svc__C': 1, 'svc__gamma': 0.1}
Validação: 82.9383%


### O melhor SVM superou a Random Forest? Por quanto?

Não. A melhor combinação foi C=1 e gamma=0.1, com validação de 82,9%. A Random Forest atingiu 83,3% e ficou cerca de 0,3 ponto percentual à frente. A diferença foi calculada antes do arredondamento das porcentagens.

A pontuação da busca foi usada para escolher os parâmetros. Ela pode ser otimista e não equivale a um teste final independente. Para uma comparação definitiva, eu separaria um conjunto de teste ou usaria validação cruzada aninhada.

## Seus resultados

In [10]:
tabela = (pd.DataFrame(resultados)
          .drop_duplicates("Modelo", keep="last")
          .sort_values("Validação", ascending=False)
          .reset_index(drop=True))
tabela.style.format({"Treino": "{:.1%}", "Validação": "{:.1%}"})

,Modelo,Treino,Validação
0,Random Forest (referência),87.5%,83.3%
1,SVM ajustado (GridSearchCV),83.4%,82.9%
2,SVM com escala,83.5%,82.5%
3,SVM C=100,86.2%,80.6%
4,SVM kernel linear,78.7%,78.7%
5,SVM gamma=10,89.8%,76.2%
6,SVM sem escala,68.2%,67.8%
7,SVM C=0.01,61.6%,61.6%


### Qual modelo eu escolheria para produção?

Eu escolheria a Random Forest como candidata inicial por ter a maior acurácia de validação neste experimento, mas confirmaria o resultado em um conjunto de teste separado antes de colocá-la em produção.